# Tips 21：自由記述を分類して要約

**業務の困りごと**：アンケートの自由記述を読み解き、傾向を報告する。

**このTipsでできること**：キーワードが含まれるか順番に調べて分類します。分類ごとの件数と例文を別シートに書き出します。

**使用ライブラリ**：pandas（生成AIへの発展は任意）

### ファイルの場所
`/content/` はColabの一時領域です。PCやGoogle Driveには自動保存されません。左側のフォルダから結果を選んでダウンロードできます。セッション終了前に保存してください。

上から順番に実行してください。各Tipsのサンプルはノートブック内で作ります。


## 1. 練習用データと準備

自由記述6件を用意します。予約・待ち時間・接客の話題が含まれています。

最後の `display()` で練習用データ全体を表として表示します。列名・行数・値を確認してから次へ進みましょう。


In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print  # Colab以外の環境向け
import pandas as pd
from pathlib import Path
base=Path('/content') if Path('/content').exists() else Path.cwd()
# 自由記述を表として用意する
responses=pd.DataFrame({'コメント':['予約が簡単で便利','待ち時間が長い','スタッフの説明が丁寧','予約しづらい','待ち時間が短くて助かる','説明が分かりにくい']})
print('自由記述（練習用データ）:')
display(responses)


## 2. 自動処理

キーワードが含まれるか順番に調べて分類します。分類ごとの件数と例文を別シートに書き出します。

**結果を見るポイント**：分類件数の合計は **6件** です。複数の話題を含む文は最初に一致したルールになります。

保存した成果物を読み直して `display()` で表示します。画面の表とダウンロードしたファイルの内容を照らし合わせてください。


In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print  # Colab以外の環境向け
# まずは決めた語で分類する。複数に該当する場合は先に見つかったテーマを採用。
# 分類名と判定に使う語の対応表
rules={'予約':['予約'],'待ち時間':['待ち'],'接客・説明':['説明','スタッフ']}
# 1つのコメントに一致する最初の分類を返す
def classify(text):
    for label, words in rules.items():
        if any(word in text for word in words):
            return label
    return 'その他'
# 全コメントに分類関数を適用する
responses['分類']=responses['コメント'].map(classify)
# 分類ごとの件数と代表例を集計する
summary=responses.groupby('分類',as_index=False).agg(件数=('コメント','size'),例=('コメント','first'))
out=base/'Tips21_自由記述分類.xlsx'
with pd.ExcelWriter(out) as writer:
    responses.to_excel(writer,'分類結果',index=False)
    summary.to_excel(writer,'集計',index=False)
print(summary)

# 分類結果と集計の両シートを確認する
for sheet_name, preview in pd.read_excel(out, sheet_name=None).items():
    print(f'【{sheet_name}】 {len(preview)} 行')
    display(preview)


## 3. 結果の確認

`assert` は条件が正しいか確認する命令です。条件が違えばそこで止まるため、想定外の集計を見つけられます。

分類件数の合計は **6件** です。複数の話題を含む文は最初に一致したルールになります。


In [ ]:
# 件数や合計が期待値と一致するか検査する
assert summary['件数'].sum()==len(responses)
print('確認OK:',out)


## 発展：生成AIで分類・要約する
Pythonで個人情報を除き、コメントを少量ずつ確認してから、許可されたAIサービスに渡します。例えば次の指示文を使います。

> 次の自由記述を「予約」「待ち時間」「接客・説明」「その他」に分類し、各分類の主な肯定意見と改善要望を1文ずつ要約してください。判定が難しいものは「要確認」とし、元のコメント番号を残してください。

AIの結果は人が抜き取り確認し、分類件数が元データと一致するか調べます。この教材は外部APIへの送信を実行しません。


## 4. 持ち帰り・練習

**練習問題**：「料金」の分類ルールを追加し、該当するコメントを作ってみましょう。

これはキーワードに基づく分類です。生成AIによる意味分類や要約は別工程で、内容の確認と個人情報の扱いが必要です。

**実務で使う前に**：これはキーワード分類です。生成AIによる意味分類・要約を実行するコードではありません。

出力をPCへ保存したい場合は次のセルをColabで実行します。Colab以外では出力場所を表示します。


In [ ]:
# Colab上で作った成果物をPCへダウンロードする
try:
    from google.colab import files
    files.download(str(out))
except ImportError:
    print('出力ファイル:',out.resolve())
